In [6]:
import os
os.environ['OPENAI_API_KEY'] = ''

In [9]:
# requirements.txt
# To run this script, install these dependencies:
# pip install -r requirements.txt

# phidata==2.7.10
# openai>=1.0.0
# duckduckgo-search>=6.1.0
# requests>=2.31.0
# argparse>=1.4.0

# # Optional for better terminal streaming / markdown output
# rich>=13.7.0


"""
Minimal Phidata agent that can use a DuckDuckGo search tool.

Setup:
  1) pip install -r requirements.txt  # phidata==2.7.10 uses the `phi` import namespace
  2) export OPENAI_API_KEY=...   # or set in code below

Run:
  python web_agent.py --q "what's new with the James Webb telescope?"

Notes:
- The Agent will decide when to call the DuckDuckGo tool based on the prompt.
- You can switch models (OpenAI, Anthropic, local via Ollama, etc.).
- For OpenAI, ensure OPENAI_API_KEY is set.
"""

import os
import argparse
from typing import Optional

# === Phidata imports (namespace is `phi` in v2.7.10) ===
from phi.agent import Agent
from phi.tools.duckduckgo import DuckDuckGo
import time

# Choose a model backend. Here we show OpenAI, but you can swap for others.
# OpenAI: requires OPENAI_API_KEY in your environment.
try:
    from phi.model.openai import OpenAIChat  # type: ignore
except Exception:
    OpenAIChat = None  # Falls back later if not available


def build_web_agent(
    model_name: Optional[str] = None,
    max_results: int = 8,
    safe_search: str = "moderate",
) -> Agent:
    """Create an Agent with the DuckDuckGo search tool attached.

    Args:
        model_name: LLM id for OpenAI (e.g., "gpt-4o-mini" or "gpt-4.1-mini"). If None, a reasonable default is used.
        max_results: How many search results DuckDuckGo should return.
        safe_search: one of {"off", "moderate", "strict"}.
    """
    # Configure the search tool
    #ddg_tool = DuckDuckGo(max_results=max_results, safe_search=safe_search)
    ddg_tool = DuckDuckGo()

    # Pick a model. If OpenAI isn't available, raise a helpful error.
    if OpenAIChat is None:
        raise RuntimeError(
            "OpenAI model bindings not available. Install phidata extras and set OPENAI_API_KEY, "
            "or swap to another model backend (e.g., phidata.model.ollama.Ollama)."
        )

    model_id = model_name or os.getenv("OPENAI_MODEL", "gpt-4o-mini")

    agent = Agent(
        name="web_search_agent",
        model=OpenAIChat(id=model_id),  # Swap to a different model class if needed
        tools=[ddg_tool],
        instructions=[
            "When searching the web, cite the sources you used with their URLs.",
            "Be concise and return a short, actionable answer first, followed by references.",
        ],
        show_tool_calls=True,   # prints tool calls for visibility
        markdown=True,          # pretty markdown output
    )
    return agent


def ask_web_agent(query: str, model_name: Optional[str] = None, max_results: int = 8, safe_search: str = "moderate", stream: bool = True) -> None:
    """Send a prompt to the web-search agent and print the response.

    Args:
        query: The user's question or instruction.
        model_name: LLM id to use (e.g., "gpt-4o-mini").
        max_results: DuckDuckGo results to retrieve.
        safe_search: One of {"off", "moderate", "strict"}.
        stream: If True, stream tokens as they arrive (better perceived latency).
    """
    agent = build_web_agent(model_name=model_name, max_results=max_results, safe_search=safe_search)
    agent.print_response(query, stream=stream)


# if __name__ == "__main__":
#     parser = argparse.ArgumentParser(description="Phidata web-search agent using DuckDuckGo tool")
#     parser.add_argument("--q", "--query", dest="query", required=True, help="Your question or search intent")
#     parser.add_argument("--no-stream", action="store_true", help="Disable token streaming")
#     parser.add_argument("--max-results", type=int, default=8, help="DuckDuckGo results to retrieve")
#     parser.add_argument("--safe-search", type=str, default="moderate", choices=["off", "moderate", "strict"],
#                         help="DuckDuckGo safe search level")
#     parser.add_argument("--model", type=str, default=None, help="OpenAI model id, e.g., gpt-4o-mini")
#     args = parser.parse_args()

#     ask_web_agent(
#         query=args.query,
#         model_name=args.model,
#         max_results=args.max_results,
#         safe_search=args.safe_search,
#         stream=not args.no_stream,
#     )

start_time = time.time()
ask_web_agent(query="what is python", model_name="gpt-4o-mini", max_results= 8)
end_time = time.time()


print(f"Execution time: {end_time - start_time:.4f} seconds")

Output()

Execution time: 6.5945 seconds
